# Lab Assignment 3 — Task 2

## Converting a CFG to Chomsky Normal Form (CNF)

Many parsing algorithms, including CKY, require the grammar to be in **Chomsky Normal Form (CNF)**.

A grammar is in CNF if its production rules have one of these forms:

1. `A -> B C` — a non-terminal produces two non-terminals.
2. `A -> 'word'` — a non-terminal produces a single terminal.

## PART 1 : The Challenge of Ambiguity & CNF Constraints

Consider:

> **I saw the man with a telescope.**

This sentence has two possible interpretations:

1. I saw a man who was holding a telescope.
2. I used a telescope to see the man.

A PCFG assigns probabilities to grammar rules. The probability of a complete parse tree is obtained by multiplying the probabilities of all production rules used in that tree.

### Your task

1. **Convert the given PCFG into strict CNF.** Identify the rule that violates CNF and replace it with equivalent binary rule(s). Preserve the probability of the original alternative.
2. **Parse the sentence using the CNF PCFG and `ViterbiParser`.** Print the most probable parse tree and its total probability.
3. **Explain mathematically why the parser selected this interpretation over the alternative.** Show the rule probabilities used in the selected tree and compare the resulting probability with the competing interpretation.

In [1]:
pcfg_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V NP PP [0.3]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

sentence = "I saw the man with a telescope"

In [2]:
import nltk
from nltk import PCFG
from nltk.parse import ViterbiParser, ChartParser
from nltk.grammar import is_terminal, is_nonterminal
from nltk.tree import Tree
from collections import Counter

### Step 1a — Find the rule that violates CNF

Instead of eyeballing the grammar, we check every production mechanically. A production is
CNF-legal only if it is `A -> B C` (two non-terminals) or `A -> 'word'` (one terminal).

In [3]:
original_grammar = PCFG.fromstring(pcfg_grammar_str)


def cnf_violations(grammar):
    """Return the productions of `grammar` that are not in Chomsky Normal Form."""
    bad = []
    for prod in grammar.productions():
        rhs = prod.rhs()
        is_binary  = len(rhs) == 2 and all(is_nonterminal(s) for s in rhs)
        is_lexical = len(rhs) == 1 and is_terminal(rhs[0])
        if not (is_binary or is_lexical):
            bad.append(prod)
    return bad


print("Original grammar is in CNF :", original_grammar.is_chomsky_normal_form())
print()
print("Productions that violate CNF:")
for prod in cnf_violations(original_grammar):
    print(f"   {prod}      <-- RHS has {len(prod.rhs())} symbols, CNF allows at most 2")

Original grammar is in CNF : False

Productions that violate CNF:
   VP -> V NP PP [0.3]      <-- RHS has 3 symbols, CNF allows at most 2


### Step 1b — Binarise `VP -> V NP PP`

The offending rule is

```text
VP -> V NP PP  [0.3]
```

Its right-hand side has **three** non-terminals. We binarise it by introducing a fresh helper
non-terminal `VP_PP` that groups the last two symbols:

```text
VP    -> V VP_PP  [0.3]
VP_PP -> NP PP    [1.0]
```

**Why the probability is preserved.** Any derivation that used the original rule now uses the two
new rules one after the other, so the probability contributed to the tree becomes

```text
P(VP -> V VP_PP) x P(VP_PP -> NP PP) = 0.3 x 1.0 = 0.3
```

which is exactly the probability of the rule it replaced. Giving the helper rule probability `1.0`
is what makes the transformation **probability-preserving**: `VP_PP` has only one way to expand, so
it can never add or remove probability mass. The distribution over `VP` still sums to
`0.7 + 0.3 = 1.0`, so the grammar remains a proper PCFG and every sentence keeps the same
probability as before.

In [4]:
# Convert the non-CNF rule:
#     VP -> V NP PP [0.3]
# into binary rules. Choose an intermediate non-terminal and
# distribute the probability so that the original derivation
# keeps the same probability.
#
#     VP    -> V VP_PP [0.3]
#     VP_PP -> NP PP   [1.0]

cnf_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V VP_PP [0.3]
    VP_PP -> NP PP [1.0]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

cnf_grammar = PCFG.fromstring(cnf_grammar_str)

### Step 1c — Verify the converted grammar

Two checks:

1. It really is in strict CNF (no production left over).
2. It is still a **proper** PCFG: for every non-terminal `A`, the probabilities of all rules
   `A -> ...` must sum to `1.0`.

In [5]:
print("CNF grammar is in CNF :", cnf_grammar.is_chomsky_normal_form())
print("Remaining violations  :", cnf_violations(cnf_grammar) or "none")
print()

print("Probability mass per non-terminal:")
mass = {}
for prod in cnf_grammar.productions():
    mass[prod.lhs()] = mass.get(prod.lhs(), 0.0) + prod.prob()

for lhs, total in mass.items():
    status = "OK" if abs(total - 1.0) < 1e-9 else "NOT NORMALISED"
    print(f"   {str(lhs):<8} sum = {total:.4f}   {status}")

CNF grammar is in CNF : True
Remaining violations  : none

Probability mass per non-terminal:
   S        sum = 1.0000   OK
   NP       sum = 1.0000   OK
   VP       sum = 1.0000   OK
   VP_PP    sum = 1.0000   OK
   PP       sum = 1.0000   OK
   Det      sum = 1.0000   OK
   N        sum = 1.0000   OK
   V        sum = 1.0000   OK
   P        sum = 1.0000   OK


### Step 2 — Parse with `ViterbiParser`

`ViterbiParser` is a probabilistic CKY parser: it fills the CKY chart but, instead of keeping
*every* label for a span, it keeps for each label only the **single most probable subtree** that
derives that span. Because the probability of a tree is the product of independent rule
probabilities, an optimal tree can only be built from optimal subtrees, so this greedy pruning is
safe. The result is the single **maximum-probability parse**.

In [6]:
words = sentence.split()
viterbi_parser = ViterbiParser(cnf_grammar)
best_trees = list(viterbi_parser.parse(words))

best_tree = best_trees[0]

print("Sentence:", sentence)
print()
print("Most probable parse tree:")
best_tree.pretty_print()
print(best_tree)
print()
print("Total probability P(tree) =", best_tree.prob())

Sentence: I saw the man with a telescope

Most probable parse tree:
     S                                    
  ___|___________                          
 |               VP                       
 |    ___________|___                      
 |   |               NP                   
 |   |        _______|____                 
 |   |       |            PP              
 |   |       |        ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 I  saw the     man with  a      telescope

(S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.0001512)

Total probability P(tree) = 0.0001512


## Analysis requirement

Do not hard-code the final probability.

Use the parse tree returned by the parser to determine which rules were selected. Then calculate the probability of the competing interpretation from the corresponding rule probabilities.

Your explanation should make clear that **ViterbiParser selects the parse with the highest product of rule probabilities**.

### Step 3a — Recompute the probability from the rules the tree actually used

`tree.productions()` gives back every production used in the tree. We look each one up in the
grammar and multiply the probabilities — this reproduces `tree.prob()` from first principles,
without typing any number by hand.

In [7]:
# Look-up table: (lhs, rhs) -> probability, built from the grammar itself.
RULE_PROB = {(p.lhs(), p.rhs()): p.prob() for p in cnf_grammar.productions()}


def tree_probability(tree, verbose=False):
    """P(tree) = product of the probabilities of every production used in the tree."""
    total = 1.0
    for prod in tree.productions():
        p = RULE_PROB[(prod.lhs(), prod.rhs())]
        total *= p
        if verbose:
            print(f"   {str(prod):<28} [{p}]")
    return total


print("Rules used by the Viterbi tree:")
recomputed = tree_probability(best_tree, verbose=True)
print()
print("Product of rule probabilities :", recomputed)
print("Probability from the parser   :", best_tree.prob())
print("They agree                    :", abs(recomputed - best_tree.prob()) < 1e-12)

Rules used by the Viterbi tree:
   S -> NP VP                   [1.0]
   NP -> 'I'                    [0.1]
   VP -> V NP                   [0.7]
   V -> 'saw'                   [1.0]
   NP -> NP PP                  [0.6]
   NP -> Det N                  [0.3]
   Det -> 'the'                 [0.8]
   N -> 'man'                   [0.5]
   PP -> P NP                   [1.0]
   P -> 'with'                  [1.0]
   NP -> Det N                  [0.3]
   Det -> 'a'                   [0.2]
   N -> 'telescope'             [0.5]

Product of rule probabilities : 0.0001512
Probability from the parser   : 0.0001512
They agree                    : True


### Step 3b — Build the competing interpretation and score it

To compare against the *other* reading we must actually obtain it. `ViterbiParser` only ever
returns the single best tree, so we use NLTK's exhaustive `ChartParser` on the same CNF grammar to
enumerate **all** parses, then score each with the same `tree_probability` function. That way the
competing probability is also computed from the grammar's rules rather than typed in.

In [8]:
all_trees = list(ChartParser(cnf_grammar).parse(words))
scored = sorted(((tree_probability(t), t) for t in all_trees), key=lambda x: -x[0])

print("Number of parses found:", len(scored))
print()
for rank, (p, t) in enumerate(scored, start=1):
    print(f"--- Parse #{rank}   P = {p:.10f} ---")
    t.pretty_print()

best_p,      best_t      = scored[0]
competing_p, competing_t = scored[1]

# ProbabilisticTree.__str__ appends "(p=...)", so compare the plain Tree structures instead.
print("Viterbi tree is the highest-scoring parse:",
      Tree.convert(best_t) == Tree.convert(best_tree))

Number of parses found: 2

--- Parse #1   P = 0.0001512000 ---
     S                                    
  ___|___________                          
 |               VP                       
 |    ___________|___                      
 |   |               NP                   
 |   |        _______|____                 
 |   |       |            PP              
 |   |       |        ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 I  saw the     man with  a      telescope

--- Parse #2   P = 0.0001080000 ---
     S                                     
  ___|___________                           
 |               VP                        
 |    ___________|____                      
 |   |              VP_PP                  
 |   |        ________|____                 
 |   |       |             PP              
 |   |       |         _

### Step 3c — Which rules actually decide the outcome?

Most of the two trees is identical, so multiplying all thirteen rules twice hides the real reason.
Below we take the **multiset difference** of the productions used by the two trees. Whatever is
common cancels out of the ratio; only the differing rules determine the winner.

In [9]:
def rule_counter(tree):
    return Counter(tree.productions())


only_best      = rule_counter(best_t)      - rule_counter(competing_t)
only_competing = rule_counter(competing_t) - rule_counter(best_t)


def show(title, rules):
    print(title)
    product = 1.0
    for prod, count in rules.items():
        p = RULE_PROB[(prod.lhs(), prod.rhs())]
        product *= p ** count
        print(f"   {str(prod):<26} [{p}]" + (f" x{count}" if count > 1 else ""))
    print(f"   -> product of the distinguishing rules = {product}")
    print()
    return product


shared = rule_counter(best_t) & rule_counter(competing_t)   # element-wise minimum

print("Rules shared by both parses (they cancel in the comparison):")
shared_product = 1.0
for prod, count in sorted(shared.items(), key=lambda kv: str(kv[0])):
    p = RULE_PROB[(prod.lhs(), prod.rhs())]
    shared_product *= p ** count
    print(f"   {str(prod):<26} [{p}]" + (f" x{count}" if count > 1 else ""))
print(f"   -> product of the shared rules = {shared_product}")
print()

p_best_diff      = show("Rules unique to the WINNING parse (NP-attachment):", only_best)
p_competing_diff = show("Rules unique to the COMPETING parse (VP-attachment):", only_competing)

print(f"Decisive ratio = {p_best_diff} / {p_competing_diff} = {p_best_diff / p_competing_diff:.4f}")
print(f"Full ratio     = {best_p} / {competing_p} = {best_p / competing_p:.4f}")
print("The two ratios match, confirming the shared rules cancel exactly.")
print()
print(f"shared x winning-diff   = {shared_product * p_best_diff:.10f}  (= P of winning parse)")
print(f"shared x competing-diff = {shared_product * p_competing_diff:.10f}  (= P of competing parse)")

Rules shared by both parses (they cancel in the comparison):
   Det -> 'a'                 [0.2]
   Det -> 'the'               [0.8]
   N -> 'man'                 [0.5]
   N -> 'telescope'           [0.5]
   NP -> 'I'                  [0.1]
   NP -> Det N                [0.3] x2
   P -> 'with'                [1.0]
   PP -> P NP                 [1.0]
   S -> NP VP                 [1.0]
   V -> 'saw'                 [1.0]
   -> product of the shared rules = 0.0003600000000000001

Rules unique to the WINNING parse (NP-attachment):
   VP -> V NP                 [0.7]
   NP -> NP PP                [0.6]
   -> product of the distinguishing rules = 0.42

Rules unique to the COMPETING parse (VP-attachment):
   VP -> V VP_PP              [0.3]
   VP_PP -> NP PP             [1.0]
   -> product of the distinguishing rules = 0.3

Decisive ratio = 0.42 / 0.3 = 1.4000
Full ratio     = 0.0001512 / 0.00010800000000000001 = 1.4000
The two ratios match, confirming the shared rules cancel exactly.

share

In [10]:
print("SUMMARY")
print("=" * 66)
print(f"Winning parse    P = {best_p:.10f}")
print(f"   {best_t}")
print()
print(f"Competing parse  P = {competing_p:.10f}")
print(f"   {competing_t}")
print()
print(f"The winning parse is {best_p / competing_p:.2f}x more probable, "
      f"so ViterbiParser returns it.")

SUMMARY
Winning parse    P = 0.0001512000
   (S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope))))))

Competing parse  P = 0.0001080000
   (S
  (NP I)
  (VP
    (V saw)
    (VP_PP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope))))))

The winning parse is 1.40x more probable, so ViterbiParser returns it.


### Step 3d — Cross-check: did the CNF conversion change anything?

If the binarisation really preserved probabilities, then parsing with the **original** (non-CNF)
grammar must give the same best probability. NLTK's `ViterbiParser` accepts n-ary rules, so we can
run it on `original_grammar` and compare.

In [11]:
original_best = list(ViterbiParser(original_grammar).parse(words))[0]

print("Original grammar  -> P =", original_best.prob())
print(original_best)
print()
print("CNF grammar       -> P =", best_tree.prob())
print(best_tree)
print()
print("Probabilities identical:", abs(original_best.prob() - best_tree.prob()) < 1e-12)
print("=> the CNF conversion is probability-preserving.")

Original grammar  -> P = 0.0001512
(S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.0001512)

CNF grammar       -> P = 0.0001512
(S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.0001512)

Probabilities identical: True
=> the CNF conversion is probability-preserving.


### Mathematical explanation

Both readings use exactly the same lexical rules and the same noun phrases, so those factors are
**identical in both products and cancel**. Written out, the shared part is

```text
S -> NP VP           [1.0]
NP -> 'I'            [0.1]
V  -> 'saw'          [1.0]
NP -> Det N          [0.3]   Det -> 'the' [0.8]   N -> 'man'       [0.5]
PP -> P NP           [1.0]   P   -> 'with' [1.0]
NP -> Det N          [0.3]   Det -> 'a'   [0.2]   N -> 'telescope' [0.5]
```

The two parses differ **only** in how `with a telescope` is attached:

**Reading 1 — NP-attachment** ("the man *who has* a telescope"): the PP hangs off the noun phrase.

```text
VP -> V NP           [0.7]
NP -> NP PP          [0.6]        product of the differing rules = 0.7 x 0.6 = 0.42
```

**Reading 2 — VP-attachment** ("saw *using* a telescope"): the PP hangs off the verb phrase, which
after binarisation goes through the helper non-terminal.

```text
VP    -> V VP_PP     [0.3]
VP_PP -> NP PP       [1.0]        product of the differing rules = 0.3 x 1.0 = 0.30
```

Since `0.42 > 0.30`, reading 1 wins, and the ratio of the *full* tree probabilities is exactly the
ratio of these two differing products:

```text
0.42 / 0.30 = 1.4
```

Multiplying the shared factors back in:

```text
shared = 1.0 x 0.1 x 1.0 x (0.3 x 0.8 x 0.5) x (1.0 x 1.0) x (0.3 x 0.2 x 0.5)
       = 0.00036

P(reading 1) = 0.00036 x 0.42 = 0.0001512
P(reading 2) = 0.00036 x 0.30 = 0.0001080
```

which are exactly the numbers the code printed above.

**Conclusion.** `ViterbiParser` does not understand telescopes or eyesight. It maximises

```text
P(tree) = prod over all rules used  P(rule)
```

and returns the single tree with the largest product. Here the grammar happens to assign a high
probability to noun-phrase modification (`NP -> NP PP` at `0.6`) combined with the plain transitive
`VP -> V NP` at `0.7`, and that combination outscores the ternary verb-phrase rule at `0.3`. The
disambiguation is therefore a purely arithmetic consequence of the rule probabilities — change
`VP -> V NP PP` from `0.3` to, say, `0.6` and the parser would flip to the instrumental reading.

# Part 2: Advanced Dependency Parsing Using spaCy

Use the **spaCy NLP library** to perform dependency parsing and graph traversal on the following sentence:

> **The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation.**

A **dependency parse** is different from the constituency parses above: there are no phrase nodes
like `NP` or `VP`. Instead every word is a node, and each word points to a single **head** word
with a labelled edge (its dependency relation). The result is a tree over the words themselves,
rooted at the main verb.

## 1. Initialize and Parse

Load the `en_core_web_sm` model in spaCy and process the sentence.

Return the parsed object and store it as **`parsed_doc`**.

In [13]:
# If the model is not installed yet, run these once:
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy
from collections import deque

nlp = spacy.load("en_core_web_sm")

target_sentence = ("The exhausted researchers at the institute discovered a new vaccine "
                   "that completely prevents the viral mutation.")

parsed_doc = nlp(target_sentence)

print(parsed_doc)
print()
print(f"{'i':<4}{'TOKEN':<14}{'POS':<8}{'DEP':<10}{'HEAD':<14}")
print("-" * 50)
for token in parsed_doc:
    print(f"{token.i:<4}{token.text:<14}{token.pos_:<8}{token.dep_:<10}{token.head.text:<14}")

ModuleNotFoundError: No module named 'spacy'

## 2. Isolate the Root

Create a function that accepts **`parsed_doc` as its only parameter**.

Iterate through the document to find the structural center of the sentence, where the dependency tag is `"ROOT"`.

Return this token and save it as **`root_node`**.

The root is the one token that is its own head — the syntactic centre that everything else
ultimately hangs from. spaCy labels it with the dependency tag `ROOT`.

In [ ]:
def find_root(parsed_doc):
    """Return the token whose dependency label is ROOT (the head of the whole sentence)."""
    for token in parsed_doc:
        if token.dep_ == "ROOT":
            return token
    return None


root_node = find_root(parsed_doc)

print("Root node :", root_node)
print("POS       :", root_node.pos_)
print("Its head  :", root_node.head, "(a root is its own head)")
print("Children  :", [f"{c.text}/{c.dep_}" for c in root_node.children])

Root node : discovered
POS       : VERB
Its head  : discovered (a root is its own head)
Children  : ['researchers/nsubj', 'vaccine/dobj', './punct']


## 3. Extract the Graph Terminals

Write a function that requires **both `root_node` and `parsed_doc`** as inputs.

1. Check the `.children` of `root_node` to find the nominal subject (`nsubj` dependency), isolating **`researchers`** as `start_node`.
2. Find the token whose text is **`mutation`** in `parsed_doc`, isolating it as `end_node`.
3. Return both nodes.

In [ ]:
def extract_terminals(root_node, parsed_doc, target_word="mutation"):
    """Return (start_node, end_node).

    start_node : the nsubj child of the root  -> 'researchers'
    end_node   : the token whose text is `target_word` -> 'mutation'
    """
    start_node = None
    for child in root_node.children:
        if child.dep_ == "nsubj":
            start_node = child
            break

    end_node = None
    for token in parsed_doc:
        if token.text == target_word:
            end_node = token
            break

    return start_node, end_node


start_node, end_node = extract_terminals(root_node, parsed_doc)

print(f"start_node : {start_node}  (dep={start_node.dep_}, head={start_node.head})")
print(f"end_node   : {end_node}  (dep={end_node.dep_}, head={end_node.head})")

start_node : researchers  (dep=nsubj, head=discovered)
end_node   : mutation  (dep=dobj, head=prevents)


## 4. Graph Traversal — Shortest Path

Treat the dependency parse as an **undirected graph**. From any token, you may move to:

- its `.head`, or
- any of its `.children`.

Write a **Breadth-First Search (BFS)** function that takes `start_node` and `end_node` as inputs.

Find the shortest path between the two words and return it as a list of tokens named **`dependency_path`**.

The dependency parse is a *directed* tree (every edge points from head to dependent), but here we
walk it as an **undirected graph** so we can climb up from `researchers` and then descend to
`mutation`.

Because BFS explores the graph one "ring" at a time, the first time it reaches `end_node` it has
necessarily done so along a path with the fewest possible edges — that is the shortest path. The
`visited` set stops us from walking back and forth along an edge forever.

In [ ]:
def bfs_shortest_path(start_node, end_node):
    """Shortest path between two tokens, treating the dependency parse as undirected.

    Returns a list of tokens from start_node to end_node (inclusive), or [] if unreachable.
    """
    if start_node is None or end_node is None:
        return []

    queue = deque([[start_node]])          # queue holds whole paths
    visited = {start_node.i}               # token.i is the unique index in the doc

    while queue:
        path = queue.popleft()
        current = path[-1]

        if current.i == end_node.i:
            return path

        # Undirected neighbours: all children, plus the head (unless we are at the root).
        neighbours = list(current.children)
        if current.head.i != current.i:
            neighbours.append(current.head)

        for nb in neighbours:
            if nb.i not in visited:
                visited.add(nb.i)
                queue.append(path + [nb])

    return []


dependency_path = bfs_shortest_path(start_node, end_node)

print("Shortest dependency path:")
print("   " + "  ->  ".join(t.text for t in dependency_path))
print()
print("Number of tokens on the path:", len(dependency_path))
print("Number of edges traversed   :", len(dependency_path) - 1)

Shortest dependency path:
   researchers  ->  discovered  ->  vaccine  ->  prevents  ->  mutation

Number of tokens on the path: 5
Number of edges traversed   : 4


## 5. Path Analysis

Write a function that accepts **`dependency_path`** as its input.

Iterate through **this list of tokens only**, not the whole sentence, and print their syntactic details in the following format:

| Word | POS | Head | Dependency |
|---|---|---|---|
| researchers | NOUN | discovered | nsubj |
| ... | ... | ... | ... |

This should extract only the most important syntactic skeleton connecting the two concepts.

In [ ]:
from IPython.display import Markdown, display


def analyse_path(dependency_path):
    """Print the syntactic details of every token on the path (and nothing else)."""
    print(f"| {'Word':<12} | {'POS':<6} | {'Head':<12} | {'Dependency':<10} |")
    print(f"|{'-' * 14}|{'-' * 8}|{'-' * 14}|{'-' * 12}|")
    for token in dependency_path:
        print(f"| {token.text:<12} | {token.pos_:<6} | {token.head.text:<12} | {token.dep_:<10} |")

    # Same table, rendered properly by the notebook.
    rows = "\n".join(f"| {t.text} | {t.pos_} | {t.head.text} | {t.dep_} |"
                     for t in dependency_path)
    display(Markdown("| Word | POS | Head | Dependency |\n|---|---|---|---|\n" + rows))


analyse_path(dependency_path)

| Word         | POS    | Head         | Dependency |
|--------------|--------|--------------|------------|
| researchers  | NOUN   | discovered   | nsubj      |
| discovered   | VERB   | discovered   | ROOT       |
| vaccine      | NOUN   | discovered   | dobj       |
| prevents     | VERB   | vaccine      | relcl      |
| mutation     | NOUN   | prevents     | dobj       |


| Word | POS | Head | Dependency |
|---|---|---|---|
| researchers | NOUN | discovered | nsubj |
| discovered | VERB | discovered | ROOT |
| vaccine | NOUN | discovered | dobj |
| prevents | VERB | vaccine | relcl |
| mutation | NOUN | prevents | dobj |

### Reading the skeleton

Following the labels down the path:

* `researchers` is the **nsubj** of `discovered` — the researchers are the ones doing the discovering;
* `vaccine` is the **dobj** of `discovered` — the thing discovered;
* `prevents` is a **relcl** (relative clause) modifying `vaccine` — "the vaccine *that … prevents …*";
* `mutation` is the **dobj** of `prevents` — the thing prevented.

So the four edges compress the whole sentence into the claim
*researchers → discovered → vaccine → prevents → mutation*. Every other word in the sentence
(`The`, `exhausted`, `at the institute`, `a`, `new`, `that`, `completely`, `the`, `viral`) is a
modifier hanging off this spine and is correctly excluded by the shortest path.

## 6. Visual Verification

Pass the original **`parsed_doc`** into spaCy's `displacy` module to render the full dependency tree.

Use the resulting diagram to visually verify that the table generated in Step 5 matches the shortest path in the dependency tree.

In [ ]:
from spacy import displacy

displacy.render(parsed_doc, style="dep", jupyter=True,
                options={"compact": True, "distance": 110, "word_spacing": 30})

In [ ]:
# If the inline diagram does not render (e.g. when running as a plain script),
# write it out as an SVG file that can be opened in a browser.
svg = displacy.render(parsed_doc, style="dep", jupyter=False,
                      options={"compact": True, "distance": 110})

with open("dependency_tree.svg", "w", encoding="utf-8") as f:
    f.write(svg)

print("Saved diagram to dependency_tree.svg")

Saved diagram to dependency_tree.svg


In [ ]:
# Verify against the diagram: the path edges below must be visible as arcs in the rendering.
print("Edges that make up the shortest path (check each one in the diagram above):\n")
for a, b in zip(dependency_path, dependency_path[1:]):
    if b.head.i == a.i:
        print(f"   {a.text}  --{b.dep_}-->  {b.text}      (downwards: {b.text} is a child of {a.text})")
    else:
        print(f"   {a.text}  --{a.dep_}-->  {b.text}      (upwards: {a.text} is a child of {b.text})")

print("\nTokens NOT on the path (modifiers, correctly skipped):")
on_path = {t.i for t in dependency_path}
print("  ", [t.text for t in parsed_doc if t.i not in on_path])

Edges that make up the shortest path (check each one in the diagram above):

   researchers  --nsubj-->  discovered      (upwards: researchers is a child of discovered)
   discovered  --dobj-->  vaccine      (downwards: vaccine is a child of discovered)
   vaccine  --relcl-->  prevents      (downwards: prevents is a child of vaccine)
   prevents  --dobj-->  mutation      (downwards: mutation is a child of prevents)

Tokens NOT on the path (modifiers, correctly skipped):
   ['The', 'exhausted', 'at', 'the', 'institute', 'a', 'new', 'that', 'completely', 'the', 'viral', '.']


### Conclusion

* **Part 1** — the only non-CNF rule was `VP -> V NP PP [0.3]`; binarising it as
  `VP -> V VP_PP [0.3]` plus `VP_PP -> NP PP [1.0]` keeps the grammar a proper PCFG and leaves every
  sentence probability unchanged (verified by parsing with both grammars).
  `ViterbiParser` picks the NP-attachment reading with `P = 0.0001512` over the VP-attachment
  reading with `P = 0.0001080`, purely because `0.7 x 0.6 = 0.42` beats `0.3 x 1.0 = 0.30` — every
  other rule is common to both trees.
* **Part 2** — treating spaCy's dependency parse as an undirected graph and running BFS gives the
  shortest path `researchers -> discovered -> vaccine -> prevents -> mutation`, a four-edge
  syntactic skeleton that captures the sentence's core claim while discarding all the modifiers.